In [1]:
# Just test PM geo 

import gc
import os
import sys
import time
from pathlib import Path

import numpy as np
import jax.numpy as jnp
import jax
from desc.io import load
#eq = load('qipm_eq.h5')
eq=load("qipm3.h5")

In [ ]:
from desc.vmec import VMECIO
VMECIO.write_vmec_input(eq,'testeq')

In [ ]:
import gc
import os
import sys
import time
from pathlib import Path
import numpy as np
import jax.numpy as jnp
import jax
from desc.io import load
from desc.dipole import import_dipoles
eq = load('qipm3.h5')
one_period = import_dipoles(3, True, 'qipm_dipoles.csv')
MAG_FILE = 'torsion_grid_circular.focus'
SURF_FILE = 'testeq'
SURFACE_RANGE = "half period"
SURFACE_NPHI   = 64
SURFACE_NTHETA = 64
full = True
save = True
def load_surface(surf_file, surface_range, nphi, ntheta):
    from simsopt.geo import SurfaceRZFourier
    try:
        surface = SurfaceRZFourier.from_vmec_input(str(surf_file), range=surface_range, nphi=nphi, ntheta=ntheta)
    except Exception:
        surface = SurfaceRZFourier.from_focus(str(surf_file), range=surface_range, nphi=nphi, ntheta=ntheta)
    return surface
def apply_symmetries(pho, positions, moments, stellsym=True, coordinate_flag='cartesian'):
    """Apply stellarator symmetries to positions and moments."""
    step = 1
    pos = positions[::step]
    mom = moments[::step]
    if coordinate_flag == 'cylindrical':
        phi_dipole = jnp.arctan2(pos[:, 1], pos[:, 0])
        mom = jnp.stack([mom[:, 0] * jnp.cos(phi_dipole) - mom[:, 1] * jnp.sin(phi_dipole),
                         mom[:, 0] * jnp.sin(phi_dipole) + mom[:, 1] * jnp.cos(phi_dipole),
                         mom[:, 2]], axis=1)
    all_pos, all_mom, all_pho = [], [], []
    stell_list = [1.0] if not stellsym else [1.0, -1.0]
    for stell in stell_list:
        pos_stell = pos * jnp.array([1.0, stell, stell])
        mom_stell = mom * jnp.array([stell, 1.0, 1.0]) if stellsym else mom
        pho_stell = pho * stell if stellsym else pho   # flip only on reflection, not per-rotation
        for i in range(2):
            angle = 2 * jnp.pi * i / 2
            R = jnp.array([[jnp.cos(angle), -jnp.sin(angle), 0.0],
                           [jnp.sin(angle), jnp.cos(angle), 0.0],
                           [0.0, 0.0, 1.0]])
            all_pos.append(pos_stell @ R.T)
            all_mom.append(mom_stell @ R.T)
            all_pho.append(pho_stell)   # same pho for every rotational copy within this stell branch
    return jnp.concatenate(all_pos, axis=0), jnp.concatenate(all_mom, axis=0), jnp.concatenate(all_pho, axis=0)
surface = load_surface(SURF_FILE, SURFACE_RANGE, SURFACE_NPHI, SURFACE_NTHETA)
mags_xyz = np.column_stack((one_period.X, one_period.Y, one_period.Z))
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=mags_xyz[:, 0],y=mags_xyz[:, 1],z=mags_xyz[:, 2],mode='markers',marker=dict(size=5,color='red',opacity = 1,),))
surface_full = load_surface(SURF_FILE, 'half period', SURFACE_NPHI, SURFACE_NTHETA)
x = surface_full.gamma()[:, :, 0]
y = surface_full.gamma()[:, :, 1]
z = surface_full.gamma()[:, :, 2]
fig.add_trace(go.Surface(x=x, y=y, z=z,opacity=1,colorbar=dict(thickness=20,len=0.75,),))
fig.update_layout(scene=dict(xaxis=dict(visible=False),yaxis=dict(visible=False),zaxis=dict(visible=False),aspectmode='data',camera=dict(eye=dict(x=1.0, y=1.0, z=1.4)),),width=1200,height=800,)
fig.show()

In [ ]:
# find maximum magnetix axis-outermost dipole distance to determine radius of coilset

import numpy as np
from scipy.spatial import cKDTree
from desc.grid import LinearGrid

# Get the magnetic axis from DESC
axis_grid = LinearGrid(
    rho=[0.0],
    M=0,
    zeta=np.linspace(0, np.pi / eq.NFP, 4096),
    NFP=eq.NFP,
)
axis_data = eq.axis.compute(["X", "Y", "Z"], grid=axis_grid)

axis_xyz = np.column_stack([
    np.asarray(axis_data["X"]).ravel(),
    np.asarray(axis_data["Y"]).ravel(),
    np.asarray(axis_data["Z"]).ravel(),
])

# Identify the outermost radial layer among active magnets.
# Assumes the focus file stores magnets ordered by tower and then layer.
# If not, use the layer indices from the grid generator instead.
tree_axis = cKDTree(axis_xyz)
distances, _ = tree_axis.query(mags_xyz, k=1)

# Approximate outermost layer as the magnets with the largest axis distance.
# This is a convenient estimate, but see note below.
outer_radius_est = distances.max()
tol = 1e-5  # m
outer_mask = distances >= outer_radius_est - tol

outer_distances = distances[outer_mask]

print(f"Maximum axis-to-dipole-center distance: {outer_distances.max():.6f} m")
print(f"Maximum axis-to-dipole-center distance: {outer_distances.max()*100:.3f} cm")
print(f"Maximum axis-to-dipole-center distance: {outer_distances.max()*1000:.3f} mm")
print(f"Magnets selected in outermost-radius band: {outer_mask.sum()}")

Maximum axis-to-dipole-center distance: 0.088023 m
Maximum axis-to-dipole-center distance: 8.802 cm
Maximum axis-to-dipole-center distance: 88.023 mm
Magnets selected in outermost-radius band: 2217


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,10))


plt.plot(mags_xyz[0::,0],mags_xyz[0::,1],'.', ms = 2, color = 'red')

plt.axis('equal')


plt.xlabel('Y',fontsize = 30)
plt.ylabel('X', fontsize = 30)
plt.grid()

In [ ]:
full_pos,full_mom,full_pho =  apply_symmetries(pho_loaded, magnet_positions, magnet_moments_raw)

In [ ]:
###
import plotly.graph_objects as go


    
fig = go.Figure()




# fig.add_trace(go.Scatter3d(
#     x=full_pos[:, 0],
#     y=full_pos[:, 1],
#     z=full_pos[:, 2],
#     mode='markers',
#     marker=dict(
#         size=5,
#         color='red',
#         opacity = 1,
#     ),
# ))


surface_full = load_surface(SURF_FILE, 'full torus', SURFACE_NPHI, SURFACE_NTHETA)

x = surface_full.gamma()[:, :, 0]
y = surface_full.gamma()[:, :, 1]
z = surface_full.gamma()[:, :, 2]

fig.add_trace(go.Surface(
    x=x, y=y, z=z,
    opacity=1,
    colorbar=dict(
        thickness=20,
        len=0.75,
    ),
))


fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data',
        camera=dict(eye=dict(x=1.0, y=1.0, z=1.4)),
    ),
    width=1200,
    height=800,
)
fig.show()

In [ ]:
import numpy as np
import pyvista as pv
import jax
import jax.numpy as jnp
from functools import partial
from scipy.optimize import minimize

from desc.dipole import _Dipole, DipoleSet, import_dipoles
from desc.io import load
from desc.grid import LinearGrid
from desc.magnetic_fields._core import dipole_field
from desc.utils import reflection_matrix, rpz2xyz, xyz2rpz, xyz2rpz_vec, rpz2xyz_vec
from desc.backend import fori_loop
fin = 'qipm_dipoles.csv'
#coilset = load('tf_coils_desc.h5')
one_period = import_dipoles(eq.NFP, eq.sym, fin)

In [ ]:
# constants throughout optimization
rhos = np.array([dip.rho for dip in one_period.dipoles])

g = DipoleSet.calc_g(one_period, eq)
coord = g[1] # xyz points to evaluate B on
coords_xyz = jnp.asarray(coord)

grid = LinearGrid(M=8, N=12, NFP=eq.NFP, sym=False, endpoint=True) # same grid used to calculate g on (default from calc_g)
n_surf = eq.surface.compute(['n_rho'], grid=grid)['n_rho']
n_surface = jnp.asarray(n_surf)

x0 = jnp.array([d.x for d in one_period.dipoles])
y0 = jnp.array([d.y for d in one_period.dipoles])
z0 = jnp.array([d.z for d in one_period.dipoles])
phi0 = jnp.array([d.phi for d in one_period.dipoles])
theta0 = jnp.array([d.theta for d in one_period.dipoles])
m0_arr = jnp.array([d.m0 for d in one_period.dipoles])
NFP = one_period.NFP
SYM = one_period.sym

B_coils_fixed = jnp.asarray(coilset.compute_magnetic_field(coord, basis="xyz"))

In [ ]:
def dipole_field_from_rho(rho, x0, y0, z0, phi0, theta0, m0_arr, coords_xyz):
    """compute B purely as a function of rho; code snippets taken from dipole.py"""
    normal = jnp.array([-jnp.sin(jnp.pi / NFP), jnp.cos(jnp.pi / NFP), 0.0])
    coords_sym = (
        coords_xyz @ reflection_matrix(normal).T @ reflection_matrix([0, 0, 1]).T
    )
    coords_xyz_full = jnp.vstack((coords_xyz, coords_sym))

    coords_rpz = xyz2rpz(coords_xyz_full)

    M0 = m0_arr * rho
    sin_theta = jnp.sin(theta0)
    m_hat = jnp.stack(
        [sin_theta * jnp.cos(phi0), sin_theta * jnp.sin(phi0), jnp.cos(theta0)],
        axis=-1,
    )
    m = M0[:, jnp.newaxis] * m_hat
    rs = jnp.stack([x0, y0, z0], axis=-1)

    def nfp_loop(k, AB):
        coords_nfp = coords_rpz + jnp.array([0, 2 * jnp.pi * k / NFP, 0])
        AB_xyz = dipole_field(rpz2xyz(coords_nfp), rs, m, chunk_size=None)
        AB = AB + xyz2rpz_vec(AB_xyz, phi=coords_nfp[:, 1])
        return AB

    AB = fori_loop(0, NFP, nfp_loop, jnp.zeros_like(coords_rpz))

    n = coords_xyz.shape[0]
    AB = AB[:n, :] + AB[n:, :] * jnp.array([-1.0, 1.0, 1.0])

    AB_xyz = rpz2xyz_vec(AB, x=coords_xyz[:, 0], y=coords_xyz[:, 1])
    return AB_xyz

def Bnormerror(rho):
    B_dipoles = dipole_field_from_rho(rho, x0, y0, z0, phi0, theta0, m0_arr, coords_xyz)
    B_total = B_dipoles + B_coils_fixed
    Bn = jnp.sum(B_total * n_surface, axis=1)
    return jnp.mean(jnp.abs(Bn))

In [ ]:
value_and_grad_fn = jax.jit(jax.value_and_grad(Bnormerror))


In [ ]:
def objective_and_grad(rho_np):
    val, grad = value_and_grad_fn(jnp.asarray(rho_np))
    val = float(val)
    return val, np.asarray(grad)

In [ ]:
bounds = [(-1.0, 1.0)] * len(rhos)

opt = minimize(
    objective_and_grad,
    x0=rhos,
    jac=True,               # objective_and_grad returns (value, grad) together
    tol=1e-12,
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 5000, "disp": True},
)
print(opt)

In [ ]:
from desc.integrals import compute_B_plasma
def compute_average_normalized_field(
    field, coils, eq, p, vacuum=False, chunk_size=None, B_plasma_chunk_size=None
):
    if B_plasma_chunk_size is None:
        B_plasma_chunk_size = chunk_size

    grid = LinearGrid(M=20, N=20, NFP=eq.NFP, endpoint=True, sym=eq.sym)

    surf_coords = eq.surface.compute(['R', 'phi', 'Z'], grid=grid)
    surf_coords_array = np.column_stack([surf_coords['R'], surf_coords['phi'], surf_coords['Z']])
    n_surf = eq.surface.compute(['n_rho'], grid=grid)['n_rho']

    B_PM    = field.compute_magnetic_field(surf_coords_array)
    B_TF    = coils.compute_magnetic_field(surf_coords_array)
    B_total = B_PM + B_TF

    Bn = np.sum(B_total * n_surf, axis=1)

    normalizing_field_vec = B_total.copy()
    if not vacuum:
        normalizing_field_vec += compute_B_plasma(
            eq, eval_grid=grid, chunk_size=B_plasma_chunk_size
        )

    normalizing_field = np.mean(np.linalg.norm(normalizing_field_vec, axis=1))

    data = eq.surface.compute(['X', 'Y', 'Z'], grid=grid)
    X = np.asarray(data['X']).reshape(41, 21)
    Y = np.asarray(data['Y']).reshape(41, 21)
    Z = np.asarray(data['Z']).reshape(41, 21)

    pgrid = pv.StructuredGrid(X, Y, Z)
    pgrid.point_data["bn"] = Bn / normalizing_field
    surf = pgrid.extract_surface()
    p.add_mesh(surf, scalars="bn", smooth_shading=True, show_edges=False)

    return np.mean(np.abs(Bn)) / normalizing_field

In [ ]:
p = pv.Plotter()

In [ ]:
one_period1=one_period.copy()
for i in range(len(opt.x)):
    one_period1[i].rho = opt.x[i]
rhos_opt = np.array([dip.rho for dip in one_period1.dipoles])

In [ ]:
print(Bnormerror(rhos)) #original discrete rho MUSE Bnormerror

In [ ]:
print(Bnormerror(rhos_opt)) #optimized continuous rho Bnormerror

In [ ]:
#mags_xyz, magnet_moments, phos